# Notebook 02 — Flow Calibration

Verifies that PoissonFlow produces event rates matching configuration.
Shows depth distribution, inter-arrival times, and book stability.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from lob_sim import LOBBook, Side
from lob_sim.flow import PoissonFlow
from lob_sim.events import Order, MarketOrder, CancelRequest

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

## 1. Seed a book and generate 5000 events

In [ ]:
LAMBDA_LIM = 5.0
LAMBDA_MKT = 1.0
LAMBDA_CANCEL = 0.3
N = 5000

book = LOBBook()
for i in range(10):
    book.submit_limit(Side.BID, 1000 - 2 - i, 10, i+1, 0.0)
    book.submit_limit(Side.ASK, 1000 + 2 + i, 10, 100+i+1, 0.0)

flow = PoissonFlow(lambda_lim=LAMBDA_LIM, lambda_mkt=LAMBDA_MKT,
                   lambda_cancel=LAMBDA_CANCEL, seed=42)

events = []
ts = 0.0
for _ in range(N):
    next_ts, ev = flow._compute_next_ts(ts, book)
    events.append((next_ts, ev))
    ts = next_ts

total_time = events[-1][0]
print(f'Generated {N} events over {total_time:.1f} seconds ({N/total_time:.2f} events/s)')

## 2. Event type fractions vs configured rates

In [ ]:
type_counts = Counter(type(ev).__name__ for _, ev in events)
print('Event type counts:')
for k, v in type_counts.items():
    print(f'  {k:20s}: {v:5d}  ({100*v/N:.1f}%)')

n_lim = type_counts['Order']
n_mkt = type_counts['MarketOrder']
n_lim_mkt = n_lim + n_mkt
obs_mkt_frac = n_mkt / n_lim_mkt if n_lim_mkt else 0
exp_mkt_frac = LAMBDA_MKT / (LAMBDA_LIM + LAMBDA_MKT)
print(f'\nMarket order fraction: observed={obs_mkt_frac:.3f}  expected={exp_mkt_frac:.3f}')

## 3. Inter-arrival time distribution

In [ ]:
times = [t for t, _ in events]
iats = np.diff(times)

fig, axes = plt.subplots(1, 2)
axes[0].hist(iats, bins=60, density=True, alpha=0.7, color='steelblue')
# Overlay theoretical Exp(total_rate)
total_rate_approx = N / total_time
x = np.linspace(0, iats.max(), 200)
axes[0].plot(x, total_rate_approx * np.exp(-total_rate_approx * x), 'r-', lw=2)
axes[0].set_xlabel('Inter-arrival time (s)')
axes[0].set_title('IAT distribution vs Exp fit')

axes[1].plot(times[:500], range(500), lw=0.8)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Cumulative events')
axes[1].set_title('Event arrival process (first 500)')

plt.tight_layout()
plt.show()

## 4. Depth distribution of limit orders

In [ ]:
limit_events = [ev for _, ev in events if isinstance(ev, Order)]
# depth = distance from BBO (abs diff from best price at their side)
# Here we approximate as geometric samples
geom_samples = [flow._geometric() for _ in range(3000)]
geom_counts = Counter(geom_samples)

depths = sorted(geom_counts.keys())
freqs = [geom_counts[d] / sum(geom_counts.values()) for d in depths]

# Theoretical Geometric(p)
p = flow.p_geom
theory = [p * (1-p)**(d-1) for d in depths]

plt.figure(figsize=(8, 4))
plt.bar(depths[:15], freqs[:15], alpha=0.7, label='Empirical')
plt.plot(depths[:15], theory[:15], 'ro-', label=f'Geometric(p={p})')
plt.xlabel('Depth (ticks from BBO)')
plt.ylabel('Probability')
plt.title('Limit order depth distribution')
plt.legend()
plt.show()
print(f'Geometric mean: {np.mean(geom_samples):.2f}  Expected: {1/p:.2f}')